In [ ]:
!pip install pyspark -q


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("WeatherWear").getOrCreate()
print("Spark running:", spark.version)

Spark running: 4.0.3


In [ ]:
import os
files = [f for f in os.listdir('/content') if f.endswith('.csv')]
print(f"Files found: {len(files)}")
print(files)

Files found: 0
[]


In [ ]:
import os
files = [f for f in os.listdir('/content') if f.endswith('.csv')]
print(f"Files found: {len(files)}")

# Also check root
all_files = os.listdir('/content')
print("All files in /content:", all_files[:10])

Files found: 0
All files in /content: ['.config', 'sample_data']


In [ ]:
import os
for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving berlin.csv to berlin.csv
Saving cairo.csv to cairo.csv
Saving cape_town.csv to cape_town.csv
Saving chicago.csv to chicago.csv
Saving dubai.csv to dubai.csv
Saving dublin.csv to dublin.csv
Saving jakarta.csv to jakarta.csv
Saving london.csv to london.csv
Saving mexico_city.csv to mexico_city.csv
Saving moscow.csv to moscow.csv
Saving mumbai.csv to mumbai.csv
Saving new_york.csv to new_york.csv
Saving paris.csv to paris.csv
Saving sao_paulo.csv to sao_paulo.csv
Saving seoul.csv to seoul.csv
Saving singapore.csv to singapore.csv
Saving stockholm.csv to stockholm.csv
Saving sydney.csv to sydney.csv
Saving tokyo.csv to tokyo.csv
Saving toronto.csv to toronto.csv


In [ ]:
import os
files_list = [f for f in os.listdir('/content') if f.endswith('.csv')]
print(f"Files found: {len(files_list)}")
print(sorted(files_list))

Files found: 20
['berlin.csv', 'cairo.csv', 'cape_town.csv', 'chicago.csv', 'dubai.csv', 'dublin.csv', 'jakarta.csv', 'london.csv', 'mexico_city.csv', 'moscow.csv', 'mumbai.csv', 'new_york.csv', 'paris.csv', 'sao_paulo.csv', 'seoul.csv', 'singapore.csv', 'stockholm.csv', 'sydney.csv', 'tokyo.csv', 'toronto.csv']


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType, TimestampType
import os

# Load all 20 CSV files
csv_files = [f"/content/{f}" for f in os.listdir('/content') if f.endswith('.csv')]
print(f"Loading {len(csv_files)} files...")

df = spark.read.csv(csv_files, header=True, inferSchema=True)
print(f"Raw rows: {df.count():,}")

# Cast types
df = df.withColumn("date", F.col("date").cast(TimestampType()))
df = df.withColumn("temperature_2m", F.col("temperature_2m").cast(FloatType()))
df = df.withColumn("precipitation", F.col("precipitation").cast(FloatType()))
df = df.withColumn("windspeed_10m", F.col("windspeed_10m").cast(FloatType()))
df = df.withColumn("relativehumidity_2m", F.col("relativehumidity_2m").cast(FloatType()))
df = df.withColumn("uv_index", F.col("uv_index").cast(FloatType()))
df = df.withColumn("weathercode", F.col("weathercode").cast(IntegerType()))

# Add year, month, season
df = df.withColumn("year", F.year("date"))
df = df.withColumn("month", F.month("date"))
df = df.withColumn("season",
    F.when(F.col("month").isin(12, 1, 2), 3)
     .when(F.col("month").isin(3, 4, 5), 0)
     .when(F.col("month").isin(6, 7, 8), 1)
     .otherwise(2)
)

# Add clothing label
df = df.withColumn("clothing_label",
    F.when((F.col("temperature_2m") > 25) & (F.col("precipitation") < 1), "Light clothing")
     .when((F.col("temperature_2m") > 15) & (F.col("precipitation") >= 1), "Waterproof clothing")
     .when((F.col("temperature_2m") > 15), "Casual wear")
     .when((F.col("temperature_2m") > 5) & (F.col("precipitation") >= 1), "Waterproof clothing")
     .when((F.col("temperature_2m") > 5), "Layered clothing")
     .when((F.col("temperature_2m") > -5), "Heavy outerwear")
     .otherwise("Thermal wear")
)

# Drop nulls
df = df.dropna(subset=["temperature_2m", "precipitation", "windspeed_10m", "relativehumidity_2m"])

print(f"Processed rows: {df.count():,}")
print("Sample:")
df.select("city", "date", "temperature_2m", "season", "clothing_label").show(5)

# Save
df.write.mode("overwrite").parquet("/content/weather_processed.parquet")
print("Saved -> /content/weather_processed.parquet")
print("Weather processing complete.")

Loading 20 files...
Raw rows: 876,480
Processed rows: 876,480
Sample:
+-----------+-------------------+--------------+------+----------------+
|       city|               date|temperature_2m|season|  clothing_label|
+-----------+-------------------+--------------+------+----------------+
|Mexico City|2019-01-01 00:00:00|         19.65|     3|     Casual wear|
|Mexico City|2019-01-01 01:00:00|         16.75|     3|     Casual wear|
|Mexico City|2019-01-01 02:00:00|          16.1|     3|     Casual wear|
|Mexico City|2019-01-01 03:00:00|         13.75|     3|Layered clothing|
|Mexico City|2019-01-01 04:00:00|         13.35|     3|Layered clothing|
+-----------+-------------------+--------------+------+----------------+
only showing top 5 rows
Saved -> /content/weather_processed.parquet
Weather processing complete.


In [ ]:
from google.colab import files
files.download('/content/weather_processed.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [ ]:
import shutil
shutil.copytree('/content/weather_processed.parquet', '/drive/MyDrive/weather_processed.parquet')
print("Copied to Google Drive!")

Copied to Google Drive!


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clothing_reviews.parquet to clothing_reviews.parquet


In [ ]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType

# Load reviews parquet
df = spark.read.parquet('/content/clothing_reviews.parquet')
print(f"Raw rows: {df.count():,}")
df.printSchema()

# Clean
df = df.withColumn("rating", F.col("rating").cast(FloatType()))
df = df.dropna(subset=["rating", "timestamp"])
df = df.filter((F.col("year") >= 2019) & (F.col("year") <= 2023))

# Map product titles to clothing categories
df = df.withColumn("clothing_category",
    F.when(F.lower(F.col("title")).rlike("jacket|coat|parka|puffer|blazer"), "Heavy outerwear")
     .when(F.lower(F.col("title")).rlike("raincoat|waterproof|rain jacket|poncho"), "Waterproof clothing")
     .when(F.lower(F.col("title")).rlike("t-shirt|tshirt|shorts|tank|bikini|swimwear|sundress"), "Light clothing")
     .when(F.lower(F.col("title")).rlike("sweater|hoodie|cardigan|fleece|pullover"), "Layered clothing")
     .when(F.lower(F.col("title")).rlike("thermal|base layer|heat|winter|snow|puffer"), "Thermal wear")
     .when(F.lower(F.col("title")).rlike("sun hat|uv|sunglasses|sun protection"), "UV-protective wear")
     .otherwise("Casual wear")
)

# Aggregate by year, month, clothing_category
df_agg = df.groupBy("year", "month", "clothing_category").agg(
    F.avg("rating").alias("avg_rating"),
    F.count("*").alias("review_count"),
)

# Add demand score — normalised review count
max_count = df_agg.agg(F.max("review_count")).collect()[0][0]
df_agg = df_agg.withColumn("demand_score",
    F.round(F.col("review_count") / max_count, 4)
)

print(f"Aggregated rows: {df_agg.count():,}")
print("Sample:")
df_agg.show(10)

# Save
df_agg.write.mode("overwrite").parquet("/content/reviews_processed.parquet")
print("Saved -> /content/reviews_processed.parquet")
print("Reviews processing complete.")

Raw rows: 326,493
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- text: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Aggregated rows: 357
Sample:
+----+-----+-------------------+-----------------+------------+------------+
|year|month|  clothing_category|       avg_rating|review_count|demand_score|
+----+-----+-------------------+-----------------+------------+------------+
|2019|    2|        Casual wear|4.350398179749716|        4395|       0.399|
|2021|    4|Waterproof clothing|4.142857142857143|           7|      6.0E-4|
|2019|   10|       Thermal wear|              4.8|          35|      0.0032|
|2023|    3|   Layered clothing|             4.44|          25|      0.0023|
|2020|    1|    Heavy outerwear|4.543103448275

In [ ]:
import shutil
shutil.copytree('/content/reviews_processed.parquet', '/drive/MyDrive/reviews_processed.parquet')
print("Copied to Google Drive!")

Copied to Google Drive!


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fashion_trends.parquet to fashion_trends.parquet


In [ ]:
import pandas as pd
from pyspark.sql import functions as F

# Load fashion parquet
df = spark.read.parquet('/content/fashion_trends.parquet')
print(f"Raw rows: {df.count():,}")
df.printSchema()

# Rename columns to match our schema
df = df.withColumnRenamed("month_of_sale", "month") \
       .withColumnRenamed("year_of_sale", "year") \
       .withColumnRenamed("sales_count", "demand_score") \
       .withColumnRenamed("category", "clothing_category")

# Encode season
df = df.withColumn("season_num",
    F.when(F.lower(F.col("season")) == "spring", 0)
     .when(F.lower(F.col("season")) == "summer", 1)
     .when(F.lower(F.col("season")) == "autumn", 2)
     .when(F.lower(F.col("season")) == "fall", 2)
     .otherwise(3)
)

# Normalise demand score
max_demand = df.agg(F.max("demand_score")).collect()[0][0]
df = df.withColumn("demand_score_norm",
    F.round(F.col("demand_score") / max_demand, 4)
)

# Keep only needed columns
df = df.select(
    "year", "month", "clothing_category", "season",
    "season_num", "demand_score", "demand_score_norm",
    "average_rating", "price"
)

print(f"Processed rows: {df.count():,}")
print("Sample:")
df.show(10)

# Save
df.write.mode("overwrite").parquet("/content/fashion_processed.parquet")
print("Saved -> /content/fashion_processed.parquet")
print("Fashion processing complete.")

Raw rows: 660
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- category: string (nullable = true)
 |-- pattern: string (nullable = true)
 |-- color: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- season: string (nullable = true)
 |-- price: double (nullable = true)
 |-- material: string (nullable = true)
 |-- sales_count: long (nullable = true)
 |-- reviews_count: long (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- out_of_stock_times: long (nullable = true)
 |-- brand: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- last_stock_date: timestamp_ntz (nullable = true)
 |-- wish_list_count: long (nullable = true)
 |-- month_of_sale: long (nullable = true)
 |-- year_of_sale: long (nullable = true)

Processed rows: 660
Sample:
+----+-----+-----------------+------+----------+------------+-----------------+--------------+------+
|year|month|cloth

In [ ]:
from pyspark.sql import functions as F

# Load all three processed datasets
weather = spark.read.parquet('/content/weather_processed.parquet')
reviews = spark.read.parquet('/content/reviews_processed.parquet')
fashion = spark.read.parquet('/content/fashion_processed.parquet')

# Aggregate weather to monthly level per city
weather_monthly = weather.groupBy("city", "year", "month", "season").agg(
    F.avg("temperature_2m").alias("avg_temperature"),
    F.avg("precipitation").alias("avg_precipitation"),
    F.avg("windspeed_10m").alias("avg_windspeed"),
    F.avg("relativehumidity_2m").alias("avg_humidity"),
    F.avg("uv_index").alias("avg_uv_index"),
    F.first("clothing_label").alias("clothing_category"),
)

print(f"Weather monthly rows: {weather_monthly.count():,}")

# Join weather with reviews on year, month, clothing_category
merged = weather_monthly.join(
    reviews,
    on=["year", "month", "clothing_category"],
    how="left"
)

print(f"After reviews join: {merged.count():,}")

# Join with fashion on year, month, clothing_category
merged = merged.join(
    fashion.select("year", "month", "clothing_category", "demand_score", "demand_score_norm", "price"),
    on=["year", "month", "clothing_category"],
    how="left"
)

print(f"Final merged rows: {merged.count():,}")
print("Sample:")
merged.show(5)

# Save
merged.write.mode("overwrite").parquet("/content/merged_dataset.parquet")
print("Saved -> /content/merged_dataset.parquet")
print("Merge complete.")

Weather monthly rows: 1,200
After reviews join: 1,200
Final merged rows: 1,200
Sample:
+----+-----+-----------------+------+------+------------------+--------------------+------------------+-----------------+------------+-----------------+------------+------------+------------+-----------------+-----+
|year|month|clothing_category|  city|season|   avg_temperature|   avg_precipitation|     avg_windspeed|     avg_humidity|avg_uv_index|       avg_rating|review_count|demand_score|demand_score|demand_score_norm|price|
+----+-----+-----------------+------+------+------------------+--------------------+------------------+-----------------+------------+-----------------+------------+------------+------------+-----------------+-----+
|2019|    1| Layered clothing|Berlin|     3|1.1145161304182263| 0.09139785088438501|16.492091920427097|  83.364968304993|        NULL|4.423728813559322|          59|      0.0054|        NULL|             NULL| NULL|
|2019|    2|  Heavy outerwear|Berlin|     3| 4.02

AnalysisException: [COLUMN_ALREADY_EXISTS] The column `demand_score` already exists. Choose another name or rename the existing column. SQLSTATE: 42711

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

# Load fashion parquet
df = spark.read.parquet('/content/fashion_trends.parquet')
print(f"Raw rows: {df.count():,}")
df.printSchema()

# Rename columns to match our schema
df = df.withColumnRenamed("month_of_sale", "month") \
       .withColumnRenamed("year_of_sale", "year") \
       .withColumnRenamed("sales_count", "demand_score") \
       .withColumnRenamed("category", "clothing_category")

# Encode season
df = df.withColumn("season_num",
    F.when(F.lower(F.col("season")) == "spring", 0)
     .when(F.lower(F.col("season")) == "summer", 1)
     .when(F.lower(F.col("season")) == "autumn", 2)
     .when(F.lower(F.col("season")) == "fall", 2)
     .otherwise(3)
)

# Normalise demand score
max_demand = df.agg(F.max("demand_score")).collect()[0][0]
df = df.withColumn("demand_score_norm",
    F.round(F.col("demand_score") / max_demand, 4)
)

# Keep only needed columns
df = df.select(
    "year", "month", "clothing_category", "season",
    "season_num", "demand_score", "demand_score_norm",
    "average_rating", "price"
)

print(f"Processed rows: {df.count():,}")
print("Sample:")
df.show(10)

# Save
df.write.mode("overwrite").parquet("/content/fashion_processed.parquet")
print("Saved -> /content/fashion_processed.parquet")
print("Fashion processing complete.")

Raw rows: 660
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- category: string (nullable = true)
 |-- pattern: string (nullable = true)
 |-- color: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- season: string (nullable = true)
 |-- price: double (nullable = true)
 |-- material: string (nullable = true)
 |-- sales_count: long (nullable = true)
 |-- reviews_count: long (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- out_of_stock_times: long (nullable = true)
 |-- brand: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- last_stock_date: timestamp_ntz (nullable = true)
 |-- wish_list_count: long (nullable = true)
 |-- month_of_sale: long (nullable = true)
 |-- year_of_sale: long (nullable = true)

Processed rows: 660
Sample:
+----+-----+-----------------+------+----------+------------+-----------------+--------------+------+
|year|month|cloth

In [ ]:
# Fix duplicate column names
merged = merged.toDF(*[c if i != merged.columns.index('demand_score', merged.columns.index('demand_score') + 1) else 'fashion_demand_score' for i, c in enumerate(merged.columns)])

# Simpler fix - select and rename explicitly
merged_clean = merged.select(
    "city", "year", "month", "season",
    "clothing_category",
    "avg_temperature", "avg_precipitation",
    "avg_windspeed", "avg_humidity", "avg_uv_index",
    "avg_rating", "review_count",
    merged["demand_score"].alias("review_demand_score"),
    "demand_score_norm", "price"
)

print(f"Final rows: {merged_clean.count():,}")
merged_clean.show(5)

merged_clean.write.mode("overwrite").parquet("/content/merged_dataset.parquet")
print("Saved -> /content/merged_dataset.parquet")
print("Merge complete.")

Final rows: 1,200
+------+----+-----+------+-----------------+------------------+--------------------+------------------+-----------------+------------+-----------------+------------+-------------------+-----------------+-----+
|  city|year|month|season|clothing_category|   avg_temperature|   avg_precipitation|     avg_windspeed|     avg_humidity|avg_uv_index|       avg_rating|review_count|review_demand_score|demand_score_norm|price|
+------+----+-----+------+-----------------+------------------+--------------------+------------------+-----------------+------------+-----------------+------------+-------------------+-----------------+-----+
|Berlin|2019|    1|     3| Layered clothing|1.1145161304182263| 0.09139785088438501|16.492091920427097|  83.364968304993|        NULL|4.423728813559322|          59|             0.0054|             NULL| NULL|
|Berlin|2019|    2|     3|  Heavy outerwear| 4.029315473678123| 0.04211309561062427|14.316028845452127|77.69526283513932|        NULL|4.578947

In [ ]:
import shutil
shutil.copytree('/content/merged_dataset.parquet', '/drive/MyDrive/merged_dataset.parquet')
print("Copied to Drive!")

Copied to Drive!


In [ ]:
shutil.copytree('/content/fashion_processed.parquet', '/drive/MyDrive/fashion_processed.parquet')
print("Fashion copied!")

Fashion copied!
